# Runtime Notebook (single file)
This notebook embeds runtime pipeline code and loads dataset/model graph from package files.

Models: `oscillator_compare`


In [ ]:
import sys, importlib.util

required = [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('matplotlib', 'matplotlib'),
    ('torch', 'torch'),
]
missing = [name for name, mod in required if importlib.util.find_spec(mod) is None]
print(f'Python: {sys.version.split()[0]}')
if missing:
    print('[warn] Missing dependencies:', ', '.join(missing))
    print('[hint] pip install ' + ' '.join(missing))
else:
    import torch
    print('[ok] Dependencies ready. torch=' + str(torch.__version__) + ', cuda=' + str(torch.cuda.is_available()))


## 1) Setup Runtime

In [ ]:
from pathlib import Path
import base64, json, sys, types

# Edit these paths before running if needed.
RUN_ROOT = '.'
NOTEBOOKS_DIR_OVERRIDE = None
DATASET_PATH_OVERRIDE = None

def _resolve_optional_path(base, p):
    if p is None:
        return None
    q = Path(str(p)).expanduser()
    return q.resolve() if q.is_absolute() else (base / q).resolve()

def _extract_drawflow_graph(payload):
    if not isinstance(payload, dict):
        return None
    if isinstance(payload.get('drawflow'), dict):
        return payload
    if isinstance(payload.get('graph'), dict):
        g = payload.get('graph')
        if isinstance(g.get('drawflow'), dict):
            return g
    model_obj = payload.get('model') if isinstance(payload.get('model'), dict) else None
    if model_obj and isinstance(model_obj.get('graph'), dict):
        g = model_obj.get('graph')
        if isinstance(g.get('drawflow'), dict):
            return g
    return None

RUN_ROOT = Path(RUN_ROOT).expanduser().resolve()
NB = _resolve_optional_path(RUN_ROOT, NOTEBOOKS_DIR_OVERRIDE) if NOTEBOOKS_DIR_OVERRIDE is not None else RUN_ROOT
resolved_dataset = _resolve_optional_path(NB, DATASET_PATH_OVERRIDE)
if resolved_dataset is not None:
    DATASET_PATH = resolved_dataset
else:
    DATASET_PATH = NB / 'dataset.csv'
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'dataset.csv not found: {DATASET_PATH}')

print('notebook runtime root:', RUN_ROOT)
print('notebook dir:', NB)
print('dataset path:', DATASET_PATH)

PIPELINE_SRC = base64.b64decode('...').decode('utf-8')
osp = types.ModuleType('oscillator_surrogate_pipeline')
osp.__file__ = 'oscillator_surrogate_pipeline.py'
sys.modules['oscillator_surrogate_pipeline'] = osp
exec(compile(PIPELINE_SRC, osp.__file__, 'exec'), osp.__dict__)
SESSIONS = json.loads(base64.b64decode('W3sic2Vzc2lvbklkIjoiY29tcGFyZV90ZXN0IiwibmFtZSI6Im9zY2lsbGF0b3JfY29tcGFyZSIsInNjaGVtYUlkIjoib3NjaWxsYXRvciIsImRhdGFzZXRTY2hlbWFJZCI6Im9zY2lsbGF0b3IiLCJtb2RlbFNjaGVtYUlkIjoib3NjaWxsYXRvciIsIm1vZGVsTmFtZSI6Im9zY2lsbGF0b3JfY29tcGFyZSIsImRhdGFzZXROYW1lIjoiZGF0YXNldF9vc2NpbGxhdG9yIiwicnVudGltZSI6InB5dGhvbl9zZXJ2ZXIiLCJydW50aW1lQmFja2VuZCI6ImF1dG8iLCJzZWVkIjo0Miwic2FtcGxlUGVyU3BsaXQiOjMsImluY2x1ZGVNb2RlbEdyYXBoIjp0cnVlLCJ0cmFpbkNmZyI6eyJlcG9jaHMiOjQwLCJiYXRjaFNpemUiOjI1NiwibGVhcm5pbmdSYXRlIjowLjAwMSwidXNlTHJTY2hlZHVsZXIiOnRydWUsImxyUGF0aWVuY2UiOjMsImxyRmFjdG9yIjowLjUsIm1pbkxyIjowLjAwMDAwMSwicmVzdG9yZUJlc3RXZWlnaHRzIjp0cnVlLCJlYXJseVN0b3BwaW5nUGF0aWVuY2UiOjB9LCJkcmF3Zmxvd0dyYXBoIjp7ImRyYXdmbG93Ijp7IkhvbWUiOnsiZGF0YSI6eyIxIjp7Im5hbWUiOiJpbnB1dF9sYXllciIsImRhdGEiOnsibW9kZSI6ImZsYXQifSwiaW5wdXRzIjp7fSwib3V0cHV0cyI6eyJvdXRwdXRfMSI6eyJjb25uZWN0aW9ucyI6W3sibm9kZSI6IjIiLCJpbnB1dCI6ImlucHV0XzEifV19fX0sIjIiOnsibmFtZSI6ImRlbnNlX2xheWVyIiwiZGF0YSI6eyJ1bml0cyI6MzIsImFjdGl2YXRpb24iOiJyZWx1In0sImlucHV0cyI6eyJpbnB1dF8xIjp7ImNvbm5lY3Rpb25zIjpbeyJub2RlIjoiMSIsIm91dHB1dCI6Im91dHB1dF8xIn1dfX0sIm91dHB1dHMiOnsib3V0cHV0XzEiOnsiY29ubmVjdGlvbnMiOlt7Im5vZGUiOiIzIiwiaW5wdXQiOiJpbnB1dF8xIn1dfX19LCIzIjp7Im5hbWUiOiJkZW5zZV9sYXllciIsImRhdGEiOnsidW5pdHMiOjE2LCJhY3RpdmF0aW9uIjoicmVsdSJ9LCJpbnB1dHMiOnsiaW5wdXRfMSI6eyJjb25uZWN0aW9ucyI6W3sibm9kZSI6IjIiLCJvdXRwdXQiOiJvdXRwdXRfMSJ9XX19LCJvdXRwdXRzIjp7Im91dHB1dF8xIjp7ImNvbm5lY3Rpb25zIjpbeyJub2RlIjoiNCIsImlucHV0IjoiaW5wdXRfMSJ9XX19fSwiNCI6eyJuYW1lIjoib3V0cHV0X2xheWVyIiwiZGF0YSI6eyJtYXRjaFdlaWdodCI6MSwidGFyZ2V0cyI6WyJ4Il0sInRhcmdldFR5cGUiOiJ4IiwibG9zcyI6Im1zZSIsInd4IjoxLCJ3diI6MX0sImlucHV0cyI6eyJpbnB1dF8xIjp7ImNvbm5lY3Rpb25zIjpbeyJub2RlIjoiMyIsIm91dHB1dCI6Im91dHB1dF8xIn1dfX0sIm91dHB1dHMiOnt9fX19fX19XQ==').decode('utf-8'))
if not isinstance(SESSIONS, list) or not SESSIONS:
    raise ValueError('SESSIONS payload is empty.')
for s in SESSIONS:
    sid = str(s.get('sessionId', s.get('name', 'session'))).strip()
    rel = str(s.get('modelGraphPath', '')).strip()
    if not rel:
        raise ValueError(f'[{sid}] missing modelGraphPath in session payload.')
    gpath = _resolve_optional_path(NB, rel)
    if gpath is None or not gpath.exists():
        raise FileNotFoundError(f'[{sid}] model graph file not found: {rel}')
    payload = json.loads(gpath.read_text(encoding='utf-8'))
    graph = _extract_drawflow_graph(payload)
    if graph is None:
        raise ValueError(f'[{sid}] invalid drawflow graph payload: {gpath}')
    s['drawflowGraph'] = graph
    s['modelGraphAbsPath'] = str(gpath)
print('sessions:', len(SESSIONS))
def _is_image_schema(schema_id):
    sid = str(schema_id or '').strip().lower()
    return sid in ('mnist', 'fashion_mnist')


## 2) Session Overview

In [ ]:
import pandas as pd

rows = []
for s in SESSIONS:
    cfg = dict(s.get('trainCfg', {}))
    rows.append({
        'session_id': s.get('sessionId', ''),
        'model_name': s.get('modelName', ''),
        'runtime': s.get('runtime', ''),
        'dataset_schema': s.get('datasetSchemaId', s.get('schemaId', '')),
        'dataset_name': s.get('datasetName', ''),
        'epochs': int(cfg.get('epochs', 40)),
        'batch_size': int(cfg.get('batchSize', 256)),
        'learning_rate': float(cfg.get('learningRate', 1e-3)),
    })
display(pd.DataFrame(rows))


## 3) Model Graphs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np

def _safe_int(v, default=0):
    try:
        return int(float(v))
    except Exception:
        return int(default)

def _extract_graph(graph):
    g = graph if isinstance(graph, dict) else {}
    data = ((g.get('drawflow', {}) or {}).get('Home', {}) or {}).get('data', {})
    if not isinstance(data, dict):
        data = {}
    ids = sorted([str(k) for k in data.keys()], key=lambda x: int(x) if str(x).isdigit() else x)
    nodes = {}
    children = {k: [] for k in ids}
    parents = {k: [] for k in ids}
    edges = []
    for nid in ids:
        n = data.get(nid, {}) if isinstance(data.get(nid, {}), dict) else {}
        x = float(n.get('pos_x', 0.0) or 0.0)
        y = float(n.get('pos_y', 0.0) or 0.0)
        name = str(n.get('name', 'node'))
        nodes[nid] = {'x': x, 'y': y, 'name': name, 'data': n.get('data', {}) if isinstance(n.get('data', {}), dict) else {}}
        outs = n.get('outputs', {}) or {}
        for ok, ov in outs.items():
            for c in ((ov or {}).get('connections', []) or []):
                to_id = str((c or {}).get('node', '')).strip()
                if not to_id or to_id not in children:
                    continue
                children[nid].append(to_id)
                parents[to_id].append(nid)
                edges.append((nid, to_id, str(ok), str((c or {}).get('input', ''))))
    return ids, nodes, edges, children, parents

def _topo_order(ids, children, parents):
    indeg = {k: len(parents.get(k, [])) for k in ids}
    q = [k for k in ids if indeg.get(k, 0) <= 0]
    q = sorted(q, key=lambda x: int(x) if str(x).isdigit() else x)
    out = []
    while q:
        cur = q.pop(0)
        out.append(cur)
        for nx in children.get(cur, []):
            indeg[nx] = int(indeg.get(nx, 0)) - 1
            if indeg[nx] == 0:
                q.append(nx)
                q.sort(key=lambda x: int(x) if str(x).isdigit() else x)
    if len(out) != len(ids):
        return ids
    return out

def _feature_dim(name, data):
    if name in ('time_sec_block', 'time_norm_block', 'sin_norm_block', 'cos_norm_block', 'ratio_km_block', 'ratio_cm_block', 'ratio_gl_block'):
        return 1
    if name == 'scenario_block':
        return 3
    if name == 'noise_schedule_block':
        return 3
    if name == 'params_block':
        pm = data.get('paramMask', {}) if isinstance(data.get('paramMask', {}), dict) else {}
        keys = ['m','c','k','e','x0','v0','gm','gk','gc','rkm','rcm','rgl']
        n = sum(1 for k in keys if bool(pm.get(k, False)))
        return int(max(1, n))
    if name in ('hist_block', 'window_hist_block'):
        fk = str(data.get('featureKey', 'x')).strip().lower()
        if fk == 'pixel':
            return 784
        return 1
    if name in ('hist_x_block', 'hist_v_block', 'x_block', 'v_block'):
        return 1
    if name in ('window_hist_x_block', 'window_hist_v_block', 'sliding_window_block'):
        w = max(1, _safe_int(data.get('windowSize', 20), 20))
        return int(w)
    return None

def _output_units_from_target(data):
    target = str(data.get('targetType', data.get('target', 'x'))).strip().lower()
    u = _safe_int(data.get('units', data.get('unitsHint', 0)), 0)
    if u > 0:
        return u
    if target == 'label':
        return 10
    if target == 'xv':
        return 2
    return 1

def _infer_node_dims(ids, nodes, children, parents):
    topo = _topo_order(ids, children, parents)
    dims = {}
    for nid in topo:
        n = nodes.get(nid, {})
        name = str(n.get('name', ''))
        data = n.get('data', {}) if isinstance(n.get('data', {}), dict) else {}
        pids = parents.get(nid, [])
        pin = [int(dims[p]) for p in pids if p in dims and dims[p] is not None]
        in_dim = sum(pin) if pin else None
        fd = _feature_dim(name, data)
        if fd is not None:
            dims[nid] = int(fd)
        elif name == 'concat_block':
            dims[nid] = int(in_dim) if in_dim is not None else None
        elif name == 'input_layer':
            dims[nid] = int(in_dim) if in_dim is not None else None
        elif name == 'dense_layer':
            dims[nid] = max(1, _safe_int(data.get('units', 32), 32))
        elif name in ('rnn_layer', 'gru_layer', 'lstm_layer'):
            dims[nid] = max(1, _safe_int(data.get('units', 64), 64))
        elif name == 'conv1d_layer':
            dims[nid] = max(1, _safe_int(data.get('filters', 64), 64))
        elif name == 'temporal_dense_layer':
            dims[nid] = max(1, _safe_int(data.get('units', 32), 32))
        elif name in ('dropout_layer', 'seq_pool_layer', 'resample_layer', 'repeat_layer'):
            dims[nid] = int(in_dim) if in_dim is not None else None
        elif name == 'output_layer':
            dims[nid] = int(_output_units_from_target(data))
        else:
            dims[nid] = int(in_dim) if in_dim is not None else None
    return dims, topo

def _node_detail(name, data, dim):
    if name == 'hist_block':
        fk = str(data.get('featureKey', 'x')).strip().lower()
        if fk == 'pixel':
            return 'hist_block=pixel -> 784 (flatten image)'
        return f'hist_block={fk} (series feature)'
    if name == 'window_hist_block':
        return f"window_hist {data.get('featureKey','x')} w={_safe_int(data.get('windowSize',20),20)}"
    if name == 'params_block':
        return f'params ({dim}d)'
    if name == 'input_layer':
        return f"input dim={dim if dim is not None else '?'}"
    if name == 'dense_layer':
        return f"Dense({_safe_int(data.get('units',32),32)})"
    if name == 'dropout_layer':
        return f"Dropout({float(data.get('rate',0.1) or 0.1):.2f})"
    if name in ('rnn_layer','gru_layer','lstm_layer'):
        return f"{name}({_safe_int(data.get('units',64),64)})"
    if name == 'conv1d_layer':
        return f"Conv1D f={_safe_int(data.get('filters',64),64)} k={_safe_int(data.get('kernelSize',3),3)}"
    if name == 'output_layer':
        target = str(data.get('targetType', data.get('target', 'x')))
        return f"Output {target}({dim if dim is not None else '?'})"
    return f"{name} ({dim if dim is not None else '?'})"

def _node_brief(name, data, dim):
    if name == 'dense_layer':
        return f"Dense({dim if dim is not None else _safe_int(data.get('units',32),32)})"
    if name == 'dropout_layer':
        return f"Drop({float(data.get('rate',0.1) or 0.1):.2f})"
    if name == 'input_layer':
        return f"Input({dim if dim is not None else '?'})"
    if name == 'output_layer':
        tgt = str(data.get('targetType', data.get('target', 'x')))
        return f"Out {tgt}({dim if dim is not None else '?'})"
    if name == 'hist_block':
        fk = str(data.get('featureKey', 'x')).strip().lower()
        if fk == 'pixel':
            return 'Hist(pixel=784)'
        return f'Hist({fk})'
    return name

def _session_model_summary(session):
    sid = str(session.get('sessionId', 'session'))
    mname = str(session.get('modelName', 'model'))
    ids, nodes, edges, children, parents = _extract_graph(session.get('drawflowGraph', {}))
    if not ids:
        return None, ids, nodes, edges, {}
    dims, topo = _infer_node_dims(ids, nodes, children, parents)
    hidden_units = []
    input_dim_total = 0
    output_units = []
    hist_note = ''
    for nid in topo:
        n = nodes[nid]
        name = str(n.get('name', ''))
        d = n.get('data', {}) if isinstance(n.get('data', {}), dict) else {}
        out_dim = dims.get(nid, None)
        if name == 'input_layer':
            if out_dim is not None:
                input_dim_total = int(max(input_dim_total, int(out_dim)))
        elif name == 'output_layer':
            if out_dim is not None:
                output_units.append(int(out_dim))
        elif name == 'hist_block':
            fk = str(d.get('featureKey', 'x')).strip().lower()
            if fk == 'pixel':
                hist_note = 'hist_block=pixel means flatten image to 784 input features'
            else:
                hist_note = f'hist_block={fk} means use {fk}(t) feature history'
        elif name in ('dense_layer','rnn_layer','gru_layer','lstm_layer','conv1d_layer','temporal_dense_layer'):
            if out_dim is not None:
                hidden_units.append(int(out_dim))
    arch = f"{input_dim_total if input_dim_total > 0 else '?'} -> {hidden_units if hidden_units else '[]'} -> {output_units if output_units else '[]'}"
    summary = {
        'session_id': sid,
        'model_name': mname,
        'input_dim': int(input_dim_total) if input_dim_total > 0 else None,
        'hidden_units': hidden_units,
        'output_units': output_units,
        'architecture': arch,
        'hist_block_note': hist_note,
    }
    return summary, ids, nodes, edges, dims

def _draw_graph(session, ids, nodes, edges, dims):
    sid = str(session.get('sessionId', 'session'))
    mname = str(session.get('modelName', 'model'))
    if not ids:
        print(f'[{sid}] no drawflowGraph found')
        return
    xs = np.asarray([nodes[k]['x'] for k in ids], dtype=np.float64)
    ys = np.asarray([nodes[k]['y'] for k in ids], dtype=np.float64)
    x0, x1 = float(xs.min()), float(xs.max())
    y0, y1 = float(ys.min()), float(ys.max())
    sx = (x1 - x0) if (x1 - x0) > 1e-9 else 1.0
    sy = (y1 - y0) if (y1 - y0) > 1e-9 else 1.0
    pos = {}
    for k in ids:
        nx = 0.06 + 0.88 * ((nodes[k]['x'] - x0) / sx)
        ny = 0.08 + 0.84 * ((nodes[k]['y'] - y0) / sy)
        pos[k] = (nx, 1.0 - ny)
    fig_w = max(10.0, min(18.0, 8.0 + 0.35 * len(ids)))
    fig_h = max(5.0, min(12.0, 4.5 + 0.22 * len(ids)))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.set_title(f'Model Graph: {mname} | session={sid}')
    for fr, to, _outk, _ink in edges:
        if fr not in pos or to not in pos:
            continue
        xA, yA = pos[fr]
        xB, yB = pos[to]
        arr = FancyArrowPatch((xA + 0.03, yA), (xB - 0.03, yB), arrowstyle='-|>', mutation_scale=12, lw=1.0, color='#64748b', alpha=0.9)
        ax.add_patch(arr)
    bw, bh = 0.13, 0.082
    for k in ids:
        x, y = pos[k]
        name = str(nodes[k]['name'])
        data = nodes[k].get('data', {}) if isinstance(nodes[k].get('data', {}), dict) else {}
        brief = _node_brief(name, data, dims.get(k, None))
        fill = '#e2e8f0'
        if name == 'input_layer':
            fill = '#bfdbfe'
        elif name == 'output_layer':
            fill = '#fecaca'
        elif 'dense' in name:
            fill = '#bbf7d0'
        box = FancyBboxPatch((x - bw / 2, y - bh / 2), bw, bh, boxstyle='round,pad=0.01,rounding_size=0.01', linewidth=1.0, edgecolor='#334155', facecolor=fill, alpha=0.96)
        ax.add_patch(box)
        txt = f"{brief}\n#{k}"
        ax.text(x, y, txt, ha='center', va='center', fontsize=7.2, color='#0f172a')
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

all_rows = []
for sess in SESSIONS:
    summary, ids, nodes, edges, dims = _session_model_summary(sess)
    if summary is not None:
        all_rows.append(summary)
    _draw_graph(sess, ids, nodes, edges, dims)
if all_rows:
    print('Model summary (input/hidden/output):')
    display(pd.DataFrame(all_rows)[['session_id','model_name','architecture','input_dim','hidden_units','output_units','hist_block_note']])


## 4) Dataset Samples

In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _parse_pixels(raw):
    if raw is None:
        return np.zeros((28, 28), dtype=np.float32)
    s = str(raw)
    vals = []
    try:
        if '[' in s and ']' in s:
            parsed = ast.literal_eval(s)
            vals = [float(x) for x in parsed]
        else:
            vals = [float(x) for x in s.split('|') if str(x).strip()]
    except Exception:
        vals = []
    if len(vals) < 784:
        vals = vals + [0.0] * (784 - len(vals))
    arr = np.asarray(vals[:784], dtype=np.float32).reshape(28, 28)
    return np.clip(arr, 0.0, 1.0)

df = pd.read_csv(DATASET_PATH)
active_schema = ''
if SESSIONS:
    active_schema = str(SESSIONS[0].get('datasetSchemaId', SESSIONS[0].get('schemaId', ''))).strip().lower()
split_counts = df['split'].value_counts().to_dict() if 'split' in df.columns else {}
display(pd.DataFrame([{'split_counts': split_counts, 'rows': int(len(df))}]))

if _is_image_schema(active_schema) and 'pixel_values' in df.columns and 'label' in df.columns:
    labels = sorted([int(x) for x in pd.Series(df['label']).dropna().unique().tolist() if str(x).strip() != ''])
    labels = labels[:10]
    for split in ['train', 'val', 'test']:
        sub = df[df['split'] == split]
        if len(sub) <= 0:
            continue
        fig, axes = plt.subplots(2, 5, figsize=(12, 5))
        axes = np.asarray(axes).reshape(-1)
        for i, lab in enumerate(labels):
            ax = axes[i]
            rows = sub[sub['label'] == lab]
            if len(rows) <= 0:
                ax.axis('off')
                ax.set_title(f'class {lab} (none)')
                continue
            row = rows.sample(1, random_state=42 + int(lab)).iloc[0]
            img = _parse_pixels(row.get('pixel_values'))
            ax.imshow(img, cmap='gray')
            ax.set_title(f"{split} | y={int(row.get('label', lab))}")
            ax.axis('off')
        for j in range(len(labels), 10):
            axes[j].axis('off')
        plt.tight_layout()
        plt.show()
else:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, split in zip(axes, ['train', 'val', 'test']):
        sub = df[df['split'] == split]
        ax.set_title(split)
        if len(sub) <= 0:
            continue
        ids = list(sub['trajectory'].dropna().astype(int).unique())[:3]
        for tid in ids:
            g = sub[sub['trajectory'] == tid].sort_values('step' if 'step' in sub.columns else 't')
            ax.plot(g.get('t', np.arange(len(g))), g.get('x', np.zeros(len(g))), lw=1.2, alpha=0.9)
        ax.set_xlabel('t')
        ax.set_ylabel('x')
    plt.tight_layout()
    plt.show()


## 5) Key Training Params and Paths

In [ ]:
import pandas as pd

# Set override values or keep None to use each session default.
EPOCHS = None
BATCH_SIZE = None
LEARNING_RATE = None

TUNE = {
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
}

for s in SESSIONS:
    cfg = dict(s.get('trainCfg', {}))
    if TUNE['epochs'] is not None:
        cfg['epochs'] = int(TUNE['epochs'])
    if TUNE['batch_size'] is not None:
        cfg['batchSize'] = int(TUNE['batch_size'])
    if TUNE['learning_rate'] is not None:
        cfg['learningRate'] = float(TUNE['learning_rate'])
    s['trainCfg'] = cfg

model_graph_paths = '; '.join([str(s.get('modelGraphPath', '')) for s in SESSIONS])
display(pd.DataFrame([
    {'key': 'RUN_ROOT', 'value': str(RUN_ROOT)},
    {'key': 'NB', 'value': str(NB)},
    {'key': 'DATASET_PATH', 'value': str(DATASET_PATH)},
    {'key': 'MODEL_GRAPH_PATHS', 'value': model_graph_paths},
]))


## 6) Train

In [ ]:
SESSION_RUNS = {}
rows = []
for s in SESSIONS:
    sid = str(s.get('sessionId', 'session'))
    cfg = dict(s.get('trainCfg', {}))
    seed = int(s.get('seed', 42))
    device_raw = str(s.get('device', 'auto')).strip().lower()
    device = None if device_raw in ('', 'auto') else device_raw
    print(f'=== train: {sid} | model={s.get("modelName", "")} ===')

    bundle = osp.build_model_and_data(
        graph_json_path=s.get('drawflowGraph', {}),
        dataset_csv_path=DATASET_PATH,
        seed=seed,
        split_mode='from_csv',
        train_frac=0.70,
        val_frac=0.15,
        test_frac=0.15,
    )
    result = osp.train_model(
        bundle,
        epochs=int(cfg.get('epochs', 40)),
        batch_size=int(cfg.get('batchSize', 256)),
        lr=float(cfg.get('learningRate', 1e-3)),
        seed=seed,
        device=device,
        use_lr_scheduler=bool(cfg.get('useLrScheduler', True)),
        scheduler_patience=int(cfg.get('lrPatience', 3)),
        scheduler_factor=float(cfg.get('lrFactor', 0.5)),
        scheduler_min_lr=float(cfg.get('minLr', 1e-6)),
        select_best_on_val=bool(cfg.get('restoreBestWeights', True)),
        early_stopping_patience=int(cfg.get('earlyStoppingPatience', 0)) if int(cfg.get('earlyStoppingPatience', 0)) > 0 else None,
        log_every=1,
    )

    SESSION_RUNS[sid] = {'session': s, 'bundle': bundle, 'result': result}
    tm = dict(result.get('test', {}))
    rows.append({
        'session_id': sid,
        'model_name': s.get('modelName', ''),
        'runtime': s.get('runtime', ''),
        'dataset_schema': s.get('datasetSchemaId', s.get('schemaId', '')),
        'test_mae': float(tm.get('mae', float('nan'))),
        'test_rmse': float(tm.get('rmse', float('nan'))),
        'test_bias': float(tm.get('bias', float('nan'))),
        'test_accuracy': float(tm.get('accuracy', float('nan'))),
        'best_epoch': result.get('best_epoch', None),
        'best_val_loss': result.get('best_val_loss', None),
    })

import pandas as pd
TRAIN_SUMMARY_DF = pd.DataFrame(rows)
display(TRAIN_SUMMARY_DF)


## 7) Epoch Report and Loss Curves

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LOSS_PLOTS = {}
HISTORY_DF = {}
for sid, pack in SESSION_RUNS.items():
    result = pack['result']
    h = dict(result.get('history', {}))
    tr = np.asarray(h.get('train_loss', []), dtype=np.float64)
    va = np.asarray(h.get('val_loss', []), dtype=np.float64)
    lr = np.asarray(h.get('lr', []), dtype=np.float64)
    m = int(max(tr.size, va.size, lr.size))
    if m <= 0:
        continue
    hist_df = pd.DataFrame({'epoch': np.arange(1, m + 1, dtype=np.int64)})
    hist_df['train_loss'] = np.pad(tr, (0, max(0, m - tr.size)), constant_values=np.nan)[:m]
    hist_df['val_loss'] = np.pad(va, (0, max(0, m - va.size)), constant_values=np.nan)[:m]
    hist_df['lr'] = np.pad(lr, (0, max(0, m - lr.size)), constant_values=np.nan)[:m]
    HISTORY_DF[sid] = hist_df
    print(f'=== epoch report: {sid} ===')
    display(hist_df)
    fig, ax1 = plt.subplots(figsize=(6, 3.2))
    ax1.plot(hist_df['epoch'], hist_df['train_loss'], label='train_loss', color='#2563eb')
    ax1.plot(hist_df['epoch'], hist_df['val_loss'], label='val_loss', color='#ef4444')
    ax1.set_xlabel('epoch')
    ax1.set_ylabel('loss')
    ax1.grid(alpha=0.25)
    ax2 = ax1.twinx()
    ax2.plot(hist_df['epoch'], hist_df['lr'], label='lr', color='#16a34a', alpha=0.8)
    ax2.set_ylabel('lr')
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='best')
    plt.tight_layout()
    plt.show()

display(pd.DataFrame([{'session_id': sid, 'history_rows': len(HISTORY_DF.get(sid, []))} for sid in SESSION_RUNS.keys()]))


## 8) Validation Plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _pred_labels_from_output(pred):
    yp = np.asarray(pred)
    if yp.ndim == 1:
        yp = yp.reshape(-1, 1)
    if yp.size == 0:
        return np.asarray([], dtype=np.int64)
    if yp.shape[1] >= 10:
        return np.argmax(yp[:, :10], axis=1).astype(np.int64)
    return np.clip(np.rint(yp[:, 0]), 0, 9).astype(np.int64)

df = pd.read_csv(DATASET_PATH)
for sid, pack in SESSION_RUNS.items():
    s = pack['session']
    result = pack['result']
    schema = str(s.get('datasetSchemaId', s.get('schemaId', ''))).strip().lower()
    print(f'=== validation plot: {sid} | schema={schema} ===')
    if _is_image_schema(schema) and 'pixel_values' in df.columns and 'label' in df.columns:
        val_df = df[df['split'] == 'val'].reset_index(drop=True)
        if len(val_df) <= 0:
            print('no val rows')
            continue
        pred_labels = _pred_labels_from_output(result.get('y_pred_val', []))
        n = min(8, len(val_df))
        fig, axes = plt.subplots(2, 4, figsize=(12, 6))
        axes = np.asarray(axes).reshape(-1)
        for i in range(n):
            row = val_df.iloc[i]
            img = _parse_pixels(row.get('pixel_values'))
            gt = int(row.get('label', -1)) if str(row.get('label', '')).strip() != '' else -1
            pred = int(pred_labels[i]) if i < len(pred_labels) else -1
            axes[i].imshow(img, cmap='gray')
            axes[i].set_title(f'GT={gt} | Pred={pred}')
            axes[i].axis('off')
        for j in range(n, len(axes)):
            axes[j].axis('off')
        plt.tight_layout()
        plt.show()
        gt_all = np.asarray(pd.to_numeric(val_df.get('label', pd.Series([], dtype=float)), errors='coerce').fillna(-1), dtype=np.int64)
        n_cm = int(min(len(gt_all), len(pred_labels)))
        if n_cm > 0:
            gt = np.clip(gt_all[:n_cm], 0, 9).astype(np.int64)
            pr = np.clip(np.asarray(pred_labels[:n_cm], dtype=np.int64), 0, 9)
            label_set = sorted(list(set(gt.tolist() + pr.tolist())))
            if not label_set:
                label_set = list(range(10))
            lut = {int(v): i for i, v in enumerate(label_set)}
            cm = np.zeros((len(label_set), len(label_set)), dtype=np.int64)
            for a, b in zip(gt, pr):
                cm[lut[int(a)], lut[int(b)]] += 1
            fig, ax = plt.subplots(figsize=(6.6, 5.8))
            im = ax.imshow(cm, cmap='Blues')
            ax.set_title(f'{sid} val confusion matrix')
            ax.set_xlabel('Predicted label')
            ax.set_ylabel('True label')
            ax.set_xticks(np.arange(len(label_set)))
            ax.set_yticks(np.arange(len(label_set)))
            ax.set_xticklabels([str(v) for v in label_set])
            ax.set_yticklabels([str(v) for v in label_set])
            vmax = int(cm.max()) if cm.size else 1
            for rr in range(cm.shape[0]):
                for cc in range(cm.shape[1]):
                    vv = int(cm[rr, cc])
                    if vv <= 0:
                        continue
                    color = 'white' if vv >= max(1, int(vmax * 0.6)) else 'black'
                    ax.text(cc, rr, str(vv), ha='center', va='center', color=color, fontsize=8)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            acc = float(np.mean(gt == pr))
            print(f'val accuracy={acc:.4f} | samples={n_cm}')
            plt.tight_layout()
            plt.show()
    else:
        y_true = np.asarray(result.get('y_true_val', []), dtype=np.float64).reshape(-1)
        y_pred = np.asarray(result.get('y_pred_val', []), dtype=np.float64).reshape(-1)
        n = int(min(len(y_true), len(y_pred), 600))
        if n <= 0:
            print('no val predictions')
            continue
        plt.figure(figsize=(9, 3.2))
        xs = np.arange(n)
        plt.plot(xs, y_true[:n], label='groundtruth', lw=1.6, alpha=0.9)
        plt.plot(xs, y_pred[:n], label='predict', lw=1.4, alpha=0.9)
        plt.title(f'{sid} val: groundtruth vs predict')
        plt.xlabel('sample index')
        plt.ylabel('value')
        plt.grid(alpha=0.25)
        plt.legend()
        plt.tight_layout()
        plt.show()


## 9) Final Report

In [ ]:
rows = []
for sid, pack in SESSION_RUNS.items():
    s = pack['session']
    result = pack['result']
    tm = dict(result.get('test', {}))
    rows.append({
        'session_id': sid,
        'model_name': s.get('modelName', ''),
        'runtime': s.get('runtime', ''),
        'dataset_schema': s.get('datasetSchemaId', s.get('schemaId', '')),
        'test_mae': float(tm.get('mae', float('nan'))),
        'test_rmse': float(tm.get('rmse', float('nan'))),
        'test_bias': float(tm.get('bias', float('nan'))),
        'test_accuracy': float(tm.get('accuracy', float('nan'))),
        'best_epoch': result.get('best_epoch', None),
        'best_val_loss': result.get('best_val_loss', None),
    })
import pandas as pd
REPORT_DF = pd.DataFrame(rows)
display(REPORT_DF)
